In [1]:
import pandas as pd
from sqlalchemy import create_engine
from datetime import datetime

In [2]:
%load_ext sql
%sql postgresql+psycopg2://ahmed@127.0.0.1:5432/postgres
%config SqlMagic.displaylimit = 100

engine = create_engine("postgresql+psycopg2://ahmed@127.0.0.1:5432/postgres")

Connecting to 'postgresql://ahmed@127.0.0.1:5432/postgres'

In [3]:
data_month_1 = pd.read_csv("2025-CSV/Data/pub0125.csv")
data_month_2 = pd.read_csv("2025-CSV/Data/pub0225.csv")
data_month_3 = pd.read_csv("2025-CSV/Data/pub0325.csv")
data_month_4 = pd.read_csv("2025-CSV/Data/pub0425.csv")
data_month_5 = pd.read_csv("2025-CSV/Data/pub0525.csv")
data_month_6 = pd.read_csv("2025-CSV/Data/pub0625.csv")
data_month_7 = pd.read_csv("2025-CSV/Data/pub0725.csv")
data_month_8 = pd.read_csv("2025-CSV/Data/pub0825.csv")
data_month_9 = pd.read_csv("2025-CSV/Data/pub0925.csv")
data_month_10 = pd.read_csv("2025-CSV/Data/pub1025.csv")
data_month_11 = pd.read_csv("2025-CSV/Data/pub1125.csv")
data_month_12 = pd.read_csv("2025-CSV/Data/pub1225.csv")

data_2025 = pd.concat([data_month_1, 
                        data_month_2, 
                        data_month_3,
                        data_month_4,
                        data_month_5,
                        data_month_6,
                        data_month_7,
                        data_month_8,
                        data_month_9,
                        data_month_10,
                        data_month_11,
                        data_month_12], 
                    ignore_index=True)

data_2025.columns = data_2025.columns.str.lower()

try:
    data_2025.to_sql(
        name = 'labour_force', 
        con = engine, 
        if_exists = 'fail', 
        index = False
    )

except ValueError as E:
    
    if ("already exists" in str(E)):
        print("Data already loaded.")

    else:
        raise

Data already loaded.


In [4]:
%%sql --save pre_weighted_data --no-execute

SELECT (hrlyearn / 100) AS "hourly_wage", 
        prov AS "province", 
        age_12 AS "age_group", 
        educ AS "education_level", 
        naics_21 AS "industry",
        firmsize AS "firm_size",
        finalwt AS "final_weight"

FROM labour_force
WHERE lfsstat IN (1, 2)
    AND hrlyearn IS NOT NULL

Running query in 'postgresql://ahmed@127.0.0.1:5432/postgres'

Skipping execution...

In [5]:
%%sql --with pre_weighted_data respondents <<

SELECT * 
FROM pre_weighted_data;

Running query in 'postgresql://ahmed@127.0.0.1:5432/postgres'

688793 rows affected.

In [6]:
respondents = respondents.DataFrame()

In [7]:
respondents.round(2).to_excel(
    f"wage_estimates_{datetime.now():%Y-%m-%d_%H-%M-%S}.xlsx",
    sheet_name = "Respondents",
    index = False
)

KeyboardInterrupt: 